In [ ]:
import pandas as pd

In [ ]:
import json

In [ ]:
import seaborn as sns

In [ ]:
from datetime import datetime

# EX 1

In [ ]:
books = pd.read_json("data/books.json")
books

In [ ]:
# Answer: how many rows and columns are in the dataset?
print(f"Shape: {books.shape}")
print(f"  Rows   : {books.shape[0]}  (one row = one book)")
print(f"  Columns: {books.shape[1]}  -> {list(books.columns)}")

In [ ]:
books[books["author"] == "Gabriel Garcia Marquez"]

In [ ]:
# Export Gabriel Garcia Marquez books to a dedicated CSV file
garcia_books = books[books["author"] == "Gabriel Garcia Marquez"]
garcia_books.to_csv("garcia_marquez_books.csv", index=False)
print(f"Exported {len(garcia_books)} book(s) to garcia_marquez_books.csv")

In [ ]:
# Remove all books by George Orwell and update the DataFrame in place
books = books.loc[books["author"] != "George Orwell"]
books

In [ ]:
books.groupby("author").agg({"title": "count"}).sort_values("title", ascending=False)

# EX 2

In [ ]:
sales = pd.read_csv("data/sales.csv")
sales

In [ ]:
sales["total"] = sales["quantity"] * sales["price"]
sales

In [ ]:
sales = sales.sort_values("total", ascending=False)
sales.head()

In [ ]:
sales.to_json("sales_exported.json")

In [ ]:
# Sum total sales per product; reset_index() promotes product_id from index to column
grouped_sales = sales.groupby("product_id").agg({"total": "sum"}).reset_index()
grouped_sales.head()

In [ ]:
sns.barplot(grouped_sales, x="product_id", y="total")

# EX3

In [ ]:
product = pd.read_json("data/product.json")
product.head()

In [ ]:
merged = pd.merge(sales, product, on="product_id", how="outer")
merged

In [ ]:
merged.info()

In [ ]:
# 'Unknown' items: product_ids present in sales but absent from product.json
# After the outer join, these rows have NaN in the 'name' column (from product).
unknown = merged[merged["name"].isna()]
print(f"Found {len(unknown)} sale row(s) with unknown product_id:")
print(unknown["product_id"].unique())

# Export the distinct unknown product IDs to a dedicated file
unknown[["product_id"]].drop_duplicates().to_csv("unknown_product_ids.csv", index=False)
print("Saved unknown product IDs to unknown_product_ids.csv")

In [ ]:
# Remove sales where the row-level total (quantity * price) is less than 300
merged = merged.loc[merged["total"] >= 300]
merged.info()

In [ ]:
# Write the combined, filtered data to aggregated_sales.csv
merged.to_csv("aggregated_sales.csv", index=False)
print(f"Saved {len(merged)} rows to aggregated_sales.csv")

In [ ]:
# Calculate total sales per product category
category_sales = (
    merged.groupby("category")
          .agg(total_sales=("total", "sum"))
          .reset_index()
          .sort_values("total_sales", ascending=False)
)
category_sales

In [ ]:
# Visualize total sales per category
sns.barplot(category_sales, x="category", y="total_sales")

In [ ]:
# Which category had the most and least total sales?
most  = category_sales.iloc[0]
least = category_sales.iloc[-1]
print(f"Most sales:  '{most['category']}'  -> {most['total_sales']:.2f}")
print(f"Least sales: '{least['category']}' -> {least['total_sales']:.2f}")

# Ex4

In [ ]:
with open("data/stations.json") as stfin:
    stations = pd.DataFrame(json.load(stfin))
stations

In [ ]:
weather = pd.read_csv("data/weather.csv")
weather

In [ ]:
weather.info()

In [ ]:
weather_merged = pd.merge(stations, weather, on="station_id")
weather_merged

In [ ]:
temp_humidity = weather_merged.groupby("station_id").agg({
    "temperature": "mean",
    "humidity": "mean"
})
temp_humidity

In [ ]:
# Daily average temperature and humidity ACROSS ALL stations
# For each calendar date we average the readings from every station.
daily_averages = (
    weather_merged
    .groupby("date")
    .agg(temperature=("temperature", "mean"),
         humidity=("humidity", "mean"))
    .reset_index()
)
daily_averages.head()

In [ ]:
# Write the daily averages to a JSON file (one record per line)
daily_averages.to_json("daily_averages.json", orient="records", indent=2)
print(f"Saved {len(daily_averages)} daily records to daily_averages.json")

In [ ]:
weather_merged["dt"] = weather_merged["date"].apply(lambda x: datetime.fromisoformat(x))
weather_merged.info()

In [ ]:
sns.lineplot(weather_merged, x="dt", y="temperature", hue="station_id")

In [ ]:
weather_merged["year-month"] = weather_merged["dt"].apply(lambda x: datetime(year=x.year, month=x.month, day=1))
weather_merged.head()

In [ ]:
weather_merged_grouped = weather_merged.groupby(["station_id", "year-month"]).agg({"temperature":"mean", "humidity":"mean"})
weather_merged_grouped.head()

In [ ]:
sns.lineplot(weather_merged_grouped, x="year-month", y="temperature", hue="station_id")

In [ ]:
sns.lineplot(weather_merged_grouped, x="year-month", y="humidity", hue="station_id")

# Ex5 — Weather Data Part 2

Using the merged weather data and the parsed `dt` datetime column from Ex4, classify each measurement by meteorological season and determine:

- Which season has the **highest average humidity** (all stations, all years)?
- Which season has the **lowest average temperature** (all stations, all years)?

Season boundaries:
- **Spring**: March 21 – June 20
- **Summer**: June 21 – September 21
- **Fall**: September 22 – December 20
- **Winter**: December 21 – March 20

In [ ]:
# Helper function: map a datetime to its meteorological season.
def get_season(dt):
    month, day = dt.month, dt.day
    # Spring: Mar 21 – Jun 20
    if (month == 3 and day >= 21) or month in [4, 5] or (month == 6 and day < 21):
        return "Spring"
    # Summer: Jun 21 – Sep 21
    elif (month == 6 and day >= 21) or month in [7, 8] or (month == 9 and day < 22):
        return "Summer"
    # Fall: Sep 22 – Dec 20
    elif (month == 9 and day >= 22) or month in [10, 11] or (month == 12 and day < 21):
        return "Fall"
    # Winter: Dec 21 – Mar 20
    else:
        return "Winter"

# Apply get_season to every row using the parsed datetime column from Ex4
weather_merged["season"] = weather_merged["dt"].apply(get_season)
weather_merged[["date", "station_id", "temperature", "humidity", "season"]].head(10)

In [ ]:
# Compute mean temperature and humidity per season (all stations, all years)
season_stats = (
    weather_merged
    .groupby("season")
    .agg(avg_temperature=("temperature", "mean"),
         avg_humidity=("humidity", "mean"))
    .reset_index()
)
season_stats

In [ ]:
# Answer: which season has the highest average humidity?
most_humid = season_stats.loc[season_stats["avg_humidity"].idxmax()]
print(f"Highest avg humidity : '{most_humid['season']}' "
      f"(avg humidity = {most_humid['avg_humidity']:.2f})")

# Answer: which season has the lowest average temperature?
coldest = season_stats.loc[season_stats["avg_temperature"].idxmin()]
print(f"Lowest avg temperature: '{coldest['season']}' "
      f"(avg temperature = {coldest['avg_temperature']:.2f})")